<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Restart-From-Hackation_v1.0/mnps_post_getting_started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Equity Post Mini-Hackathon
> A notebook to help you get started  
> DSI DSSG + MNPS Hackathon  
> September 9, 2025  
> Drafted by Wayne Birch - [contact her](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).

Details from the Hacakation follow:

You aren't constrained to what is in this notebook, and please feel free to use your creativity to deliver the best solution

## **1** | Competition Parameters
* **Outcome and evaluation**: Participants will be evaluated on the performance of their provided solution on the holdout set. Importantly, judges must be able to easily run the submitted code on the new dataset.
* **Objective**: The overall objective is to create a system which best automatically, reproducibly, and reliably categorizes jobs according to the parameters set forth by MNPS. A few suggestions are provided on parameters that you can vary if you're thinking about achievable changes in 2.5 hours


## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.

In [1]:
!pip install openai

In [12]:
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


🔧 OPENAI_MODEL (raw): GPT-4o
✅ Using MODEL_ID: gpt-4o-2024-11-20


In [13]:
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


👍 gpt-4o-2024-11-20 is available.


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.


In [14]:
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip
replace /content/Korn_Ferry Lominger 38 Competencies.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/Korn_Ferry Lominger 38 Competencies.csv  
  inflating: /content/Competency Extended Descriptions.csv  
  inflating: /content/MNPS KSACs.csv  
  inflating: /content/MNPS Roles.csv  


In [ ]:
# ===== Output folder on Google Drive =====
from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
import json

# Mount Drive (you'll be prompted once)
drive.mount('/content/drive')

# Where to save results (matches your structure)
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# New timestamped run folder, e.g., 20250909_175157
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / RUN_STAMP
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Saving outputs to:", RUN_DIR)


In [15]:
import pandas as pd
resources_dir_prefix = '/content/'
roles_lookup = pd.read_csv(resources_dir_prefix+"MNPS Roles.csv")
determinants = pd.read_csv(resources_dir_prefix+"Competency Extended Descriptions.csv", encoding='latin1')
ksac_table = pd.read_csv(resources_dir_prefix+"MNPS KSACs.csv")
korn_ferry = pd.read_csv(resources_dir_prefix+"Korn_Ferry Lominger 38 Competencies.csv", encoding='latin1')

ground_truth_masterfile = pd.read_csv(f"{base_target_folder}/Ground Truth Masterfile.csv", encoding='latin1')
new_sample = pd.read_csv(f"{base_target_folder}/New Sample_08.07.2025.csv", encoding='latin1')

## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [16]:
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

Instead of asking for a table output, we will use **structured outputs**. Though this is a common approach for the outputs of LLMs/AI systems, you can learn more about this on [OpenAI's structured output documentation](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses). Note that you can find this information on almost all LLM/AI platform or package providers.

In [17]:
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [18]:
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

In [24]:
# ===== Create classifications (Structured Outputs, with visible prints) =====
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
import json

# ---- Prompt (leave as-is unless you want to tweak wording) ----
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., Specialist I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

# ---- Pydantic schema for Structured Outputs ----
class JobClassification(BaseModel):
    job_title_original: str = Field(..., description="Original job title using the specified convention.")
    new_job_title: str = Field(..., description="Proposed new job title using the specified convention.")
    major_role_group: str = Field(..., description="Major functional group (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="Minor sub-group/level (e.g., I, II, III, IV).")
    grouping_justification: str = Field(..., description="Justification referencing job attributes and documents.")

class JobClassificationTable(BaseModel):
    job_classification_table: List[JobClassification] = Field(..., description="Table of job classifications.")
    narrative_rationale: str = Field(..., description="Narrative commentary on the overall groupings.")

# ---- Single client (API key already set in your Environment Setup cell) ----
client = OpenAI()

# TODO: Replace with your actual job text (or a variable you constructed earlier)
job_desc_text = "[Paste Job Description Here]"

# ---- Use supported roles: system + user ----
messages = [
    {"role": "system", "content": zero_shot_prompt},
    {"role": "user", "content": f"Classify the following job description:\n\n{job_desc_text}"}
]

# ---- Structured Outputs call ----
response = client.beta.chat.completions.parse(
    model=MODEL_ID,                   # comes from your Environment Setup cell
    messages=messages,
    temperature=1,
    max_tokens=1000,
    response_format=JobClassificationTable
)

# response = client.beta.chat.completions.parse(...)

msg = response.choices[0].message
raw_json_str = msg.content or ""
parsed_obj = getattr(msg, "parsed", None)

# Always save the raw JSON string (useful for debugging)
(raw_path := RUN_DIR / "Raw_Response.json").write_text(raw_json_str, encoding="utf-8")

if parsed_obj is not None:
    # 1) Table → CSV
    rows = [row.model_dump() for row in parsed_obj.job_classification_table]
    df = pd.DataFrame(rows)
    csv_path = RUN_DIR / "Job_Classifications.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")

    # 2) Narrative → TXT
    (RUN_DIR / "Narrative.txt").write_text(parsed_obj.narrative_rationale, encoding="utf-8")

    print("Wrote:")
    print(" •", csv_path)
    print(" •", RUN_DIR / "Narrative.txt")
    print(" •", raw_path)
else:
    print("No parsed Structured Output returned; saved Raw_Response.json only at:", raw_path)

# List what's in the folder so you can verify
print("\nContents of run folder:")
for p in RUN_DIR.glob("*"):
    print(" -", p.name)

# ---- SHOW THE OUTPUTS (raw + parsed) ----
msg = response.choices[0].message

print("\n=== RAW JSON STRING FROM MODEL ===")
print(msg.content or "(empty)")

if hasattr(msg, "parsed") and msg.parsed is not None:
    print("\n=== PARSED (Pydantic object) ===")
    print(msg.parsed.model_dump_json(indent=2))
else:
    # Fallback: try to pretty-print the raw content as JSON if possible
    try:
        print("\n=== RAW content pretty-printed as JSON ===")
        print(json.dumps(json.loads(msg.content), indent=2))
    except Exception:
        print("\n(No parsed object returned; see RAW JSON STRING above.)")



=== RAW JSON STRING FROM MODEL ===
{"job_classification_table":[{"job_title_original":"Data Analyst","new_job_title":"Data Analyst I","major_role_group":"Analyst","minor_sub_group":"I","grouping_justification":"The responsibilities focus on data interpretation and reporting, typical of an Analyst role at an entry level."}],"narrative_rationale":"Groupings were based on the detailed analysis of the job description's tasks, responsibilities, and required competencies. The job emphasizes foundational analytical skills, justifying its classification under the Analyst group, at an entry (I) level."}

=== PARSED (Pydantic object) ===
{
  "job_classification_table": [
    {
      "job_title_original": "Data Analyst",
      "new_job_title": "Data Analyst I",
      "major_role_group": "Analyst",
      "minor_sub_group": "I",
      "grouping_justification": "The responsibilities focus on data interpretation and reporting, typical of an Analyst role at an entry level."
    }
  ],
  "narrative_ra

In [25]:
#look at response
response.choices[0].message.parsed

JobClassificationTable(job_classification_table=[JobClassification(job_title_original='Data Analyst', new_job_title='Data Analyst I', major_role_group='Analyst', minor_sub_group='I', grouping_justification='The responsibilities focus on data interpretation and reporting, typical of an Analyst role at an entry level.')], narrative_rationale="Groupings were based on the detailed analysis of the job description's tasks, responsibilities, and required competencies. The job emphasizes foundational analytical skills, justifying its classification under the Analyst group, at an entry (I) level.")

We can make this into a table using pandas!

In [26]:
response_dict = dict(*response.choices[0].message.parsed.job_classification_table)
response_dict

{'job_title_original': 'Data Analyst',
 'new_job_title': 'Data Analyst I',
 'major_role_group': 'Analyst',
 'minor_sub_group': 'I',
 'grouping_justification': 'The responsibilities focus on data interpretation and reporting, typical of an Analyst role at an entry level.'}

In [27]:
# see outputs
pd.DataFrame(response_dict, index=[0])

,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification
0,Data Analyst,Data Analyst I,Analyst,I,The responsibilities focus on data interpretat...


In [28]:
import os
import datetime
import shutil
import ipykernel

# Get the notebook name
try:
    # This method works in Colab
    notebook_path = ipykernel.get_connection_file()
    notebook_name = os.path.basename(notebook_path).split('.')[0]
except:
    # Fallback for other environments
    notebook_name = 'Colab_Notebook_Run'

# Define the destination directory in Google Drive
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
destination_dir = f'/content/drive/My Drive/Colab Notebooks/Run Results/{timestamp}'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# List of files to copy (modify this list as needed)
files_to_copy = [
    '/content/Korn_Ferry Lominger 38 Competencies.csv',
    '/content/Competency Extended Descriptions.csv',
    '/content/MNPS KSACs.csv',
    '/content/MNPS Roles.csv',
    f'{base_target_folder}/Ground Truth Masterfile.csv', # Copying from the original location
    f'{base_target_folder}/New Sample_08.07.2025.csv', # Copying from the original location
    # Add any other files you want to copy from the run, e.g., output files
    # '/content/your_output_file.csv'
]

# Copy the files
for file_path in files_to_copy:
    try:
        shutil.copy(file_path, destination_dir)
        print(f"Copied: {file_path} to {destination_dir}")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error copying {file_path}: {e}")

print("File copying complete.")

Copied: /content/Korn_Ferry Lominger 38 Competencies.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_175157
Copied: /content/Competency Extended Descriptions.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_175157
Copied: /content/MNPS KSACs.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_175157
Copied: /content/MNPS Roles.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_175157
Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Ground Truth Masterfile.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_175157
Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/New Sample_08.07.2025.csv to /content/drive/My Drive/Colab Notebooks/Run Results/20250909_175157
File copying complete.
